# Risk composition: what has data behind it, and what does not

`Risk = Hazard x Exposure x Vulnerability` (`src/argotech/domain/risk.py`, the IPCC AR5/AR6 risk
framing -- **[verified]** as a body of work, no single chapter cited; see
[docs/CONCEPTS.md, Part 3](../docs/CONCEPTS.md)).
This notebook checks each term against the committed panel (`data/training_set.parquet`, 4,596
rows) and reports what it finds, computed below rather than asserted. A **Sources** section at the
end (section 6) collects every citation used here; this notebook does not duplicate
`docs/CONCEPTS.md`'s bibliography, it cross-references it.

* **Hazard** (`water_satisfaction`, `dry_spell_days`, `heat_days`, `vegetation`) -- the panel carries
  all of these. Section 2 computes it for every row with the real `domain.risk.assess_hazard`.
* **Exposure** (`area_ha`, `expected_yield_t_ha`, `price_per_t`) -- **zero** columns in the panel.
* **Vulnerability** (survey `signals`: assets, credit, extension access, ...) -- **zero** columns in
  the panel.

There are no farmer- or household-level columns anywhere in the 4,596-row panel. That data reaches
the service only at serving time, from the Kotlin backend's database (`docs/FACTS.md`, section on
holding-level data: `farmer_profiles`, `farms`, `farm_crops`, `crops`, `farmers_ml_profiles` feed
"the exposure and vulnerability terms of the risk composition only"). None of it enters the trained
model.

So **two of the three terms that multiply together into a risk score have never been evaluated
against any data in this repository**, while the product's triage queue is ordered by
`expected_loss = exposure x loss_rate` -- a quantity that leans on the term with the least evidence.
Section 4 makes that concrete.

This notebook reads `../data/training_set.parquet` and `../experiments/csv/*.csv` only -- no
re-fetch, no write, and every formula is imported from `argotech.domain`, never reimplemented.

## 1. Setup

In [1]:
import matplotlib
matplotlib.use("Agg")

import inspect
import pandas as pd, numpy as np, matplotlib.pyplot as plt

pd.set_option('display.width', 140); pd.set_option('display.max_columns', 50)

from argotech.domain import risk

panel = pd.read_parquet('../data/training_set.parquet')
panel['obs_date'] = pd.to_datetime(panel['obs_date'])
panel['label_date'] = pd.to_datetime(panel['label_date'])
print(panel.shape)
panel.head(3)

(4596, 35)


,gdd_90,gdd_since_onset,tmax_mean_30,tmin_mean_30,diurnal_range_30,heat_stress_days,rain_30,rain_90,et0_90,water_satisfaction_30,water_deficit_30,dry_spell_30,dry_spell_90,rain_anomaly_30,rh_mean_30,radiation_90,days_since_onset,stage_kc,ndvi,ndmi,evi,vci,ndvi_z_peer,rvi,vh_vv_ratio,rvi_z_peer,latitude,longitude,elevation,label,forward_z,site_id,cluster,obs_date,label_date
0,1231.8,1231.8,28.34,20.13,8.21,0,162.2,603.8,310.0,1.000,0.0,5,5,10.9,83.5,1550.4,90,1.05,0.7424,0.2260,0.5172,50.0,1.511791,0.6664,0.1999,-0.188156,11.3375,7.9667,693.0,0,0.1861,Kaduna_Grain_Belt-000,Kaduna_Grain_Belt,2022-10-10,2022-11-09
1,1244.3,1231.3,29.77,18.76,11.01,0,0.6,391.7,380.5,0.003,179.4,30,32,-10.3,49.3,1733.4,89,1.05,0.4186,-0.0193,0.2799,50.0,0.401608,0.4408,0.1238,-2.418947,11.3375,7.9667,693.0,1,-0.4372,Kaduna_Grain_Belt-000,Kaduna_Grain_Belt,2022-11-09,2022-12-09
2,1245.0,1245.0,29.35,17.07,12.27,0,0.0,162.8,474.7,0.000,195.6,30,62,-0.4,31.5,1918.2,90,1.05,0.2431,-0.1327,0.1643,0.0,-0.346933,0.3141,0.0852,-2.192423,11.3375,7.9667,693.0,0,-0.1190,Kaduna_Grain_Belt-000,Kaduna_Grain_Belt,2022-12-09,2023-01-08


### Verifying the table above: panel columns against each function's signature

Not asserted -- checked. `inspect.signature` reads the real parameter lists straight out of
`domain/risk.py`; the set comparison against `panel.columns` is what the opening table is built on.

In [2]:
hazard_params = list(inspect.signature(risk.assess_hazard).parameters)
exposure_params = list(inspect.signature(risk.assess_exposure).parameters)
vulnerability_params = list(inspect.signature(risk.assess_vulnerability).parameters)

panel_cols = set(panel.columns)

print("assess_hazard params:", hazard_params)
print("  -> water_satisfaction_30 in panel:", 'water_satisfaction_30' in panel_cols)
print("  -> dry_spell_30 in panel:         ", 'dry_spell_30' in panel_cols)
print("  -> heat_stress_days in panel:     ", 'heat_stress_days' in panel_cols)
print("  -> vegetation source (ndvi_z_peer) in panel:", 'ndvi_z_peer' in panel_cols)
print("  -> cumulative_dsv (any form) in panel:      ", any('dsv' in c for c in panel_cols))
print()
print("assess_exposure params:", exposure_params, "-> any present in panel:",
      any(p in panel_cols for p in exposure_params))
print("assess_vulnerability params:", vulnerability_params,
      "(dict of survey signals) -> any coping-factor column in panel:",
      any(f in panel_cols for f in risk.COPING_FACTORS))

assess_hazard params: ['water_satisfaction', 'dry_spell_days', 'cumulative_dsv', 'heat_days', 'vegetation', 'spray_threshold']
  -> water_satisfaction_30 in panel: True
  -> dry_spell_30 in panel:          True
  -> heat_stress_days in panel:      True
  -> vegetation source (ndvi_z_peer) in panel: True
  -> cumulative_dsv (any form) in panel:       False

assess_exposure params: ['area_ha', 'expected_yield_t_ha', 'price_per_t'] -> any present in panel: False
assess_vulnerability params: ['signals'] (dict of survey signals) -> any coping-factor column in panel: False


**One qualification the opening table glosses over.** `assess_hazard` takes four required
arguments, not three -- `cumulative_dsv` (accumulated late-blight Disease Severity Value) is the
fourth. Production computes it at serving time from an hourly (temperature, wet-hours) series
fetched fresh (`serving/pipeline.py:92-98, 188`); that series is daily/sub-daily raw weather, and it
was never retained as a column when the panel was built -- only 30/90-day aggregates were. So it is
not "yes, all of them" without an asterisk: **three of the four hazard sub-components have panel
data (water, dry spell, heat, vegetation); the fourth (disease) does not, in this offline
reconstruction.** Section 2 sets `cumulative_dsv=0` for every row and says so at the point it does
it -- that is a limitation of recomputing hazard from the committed panel after the fact, not of the
serving pipeline, which does have the raw series when it runs.

## 1b. The chain, end to end

What actually flows, by file and function -- no summary, the real call graph:

1. **Weather** -- Open-Meteo daily variables (`data/meteo.py`), and **satellite** -- Sentinel-2
   optical indices and Sentinel-1 radar (`data/sentinel.py`).
2. **`features/agronomic.py:build`** -- the single feature builder shared by training and serving.
   Delegates every computation to `domain/agronomy.py` and `domain/indices.py`, which are pure
   functions with no training data behind them. `domain/indices.py` is where the satellite indices
   in the panel (`ndvi`, `ndmi`, `evi`, `vci` -- all four visible in the `panel.head()` above) are
   computed:
   - `vci` -- Vegetation Condition Index, current NDVI placed within the field's own historical
     range. Kogan, F.N. (1990), *International Journal of Remote Sensing* 11:1405-1419,
     [doi:10.1080/01431169008955102](https://doi.org/10.1080/01431169008955102) --
     **[verified]**, formula identical to the implementation.
   - `evi` -- Enhanced Vegetation Index, soil/aerosol-corrected greenness. Huete et al. (2002),
     *Remote Sensing of Environment* 83:195-213 -- **[verified]**, the implemented coefficients
     (G=2.5, C1=6, C2=7.5, L=1) are exactly the published MODIS values.
   - `savi` -- Soil-Adjusted Vegetation Index, implemented in `domain/indices.py` but not among the
     panel's `FEATURE_COLUMNS` -- defined, not wired into this dataset. Huete, A.R. (1988), *Remote
     Sensing of Environment* 25:295-309 -- **[verified]**, including the `L=0.5` default.

   All three: see [docs/CONCEPTS.md, Part 1](../docs/CONCEPTS.md) for the full formula and warrant.
3. **`domain/agronomy.py`**:
   - `growing_degree_days` -- thermal time (-> `gdd_90`, `gdd_since_onset`). Standard base-10C maize
     scale; McMaster & Wilhelm (1997) is the usual methods reference for the cap variant --
     **[UNVERIFIED]** (`docs/CONCEPTS.md`, Part 2).
   - `water_balance` -- FAO-56 crop water balance (-> `water_satisfaction_30`, `water_deficit_30`,
     `longest_dry_spell_days` -> `dry_spell_30`/`dry_spell_90`). Allen, R.G., Pereira, L.S., Raes,
     D., Smith, M. (1998), *Crop Evapotranspiration -- Guidelines for Computing Crop Water
     Requirements*, FAO Irrigation and Drainage Paper 56 -- **[verified]**, authors, title and the
     single-Kc approach confirmed (`docs/CONCEPTS.md`, Part 2).
   - `heat_stress_days` -- days above the pollen-viability threshold during flowering
     (-> `heat_stress_days`)
   - `daily_severity_value` / `accumulate_dsv` -- Wallin, J.R. (1962), severity values from duration
     of RH >= 90% and mean temperature; BLITECAST (Krause, Massie & Hyre 1975) combines Wallin's
     values with Hyre's favourable-days rule; the 18-20 SV spray threshold is Wallin's --
     **[verified]**, including the threshold value the code uses (`docs/CONCEPTS.md`, Part 2).
     **Deviation to record**: the implementation drives DSV from leaf-wetness *hours*, where Wallin
     specified RH >= 90% *duration* -- a common substitution in the literature, but a substitution.
     Computed at serving time only (`dsv_total`, per the note above).
   - `fall_armyworm_generations` -- generations = cumulative GDD (base 10C) / 390. Present in
     `domain/agronomy.py` as part of the same module; not called anywhere in this notebook's hazard
     reconstruction, since `assess_hazard` has no pest term. ~390 DD/generation --
     **[UNVERIFIED]** (`docs/CONCEPTS.md`, Part 2).
4. **`domain/risk.py`**:
   - `assess_hazard(water_satisfaction, dry_spell_days, cumulative_dsv, heat_days, vegetation)` ->
     `Hazard`, combined via `noisy_or`
   - `assess_exposure(area_ha, expected_yield_t_ha, price_per_t)` -> `Exposure`
   - `assess_vulnerability(signals)` -> `Vulnerability`
   - `assess_risk(hazard, exposure, vulnerability)` -> `RiskAssessment` (`risk_score`,
     `expected_loss`)

`serving/pipeline.py` is where all three terms actually get composed for a live request
(`risk.assess_hazard(...)` at line 330, `risk.assess_exposure(...)` at line 335,
`risk.assess_vulnerability(...)` at line 342, `risk.assess_risk(...)` at line 343) -- exposure and
vulnerability's inputs come from `farmer`/`self._coping_signals(farmer)`, objects that exist only in
that request, never in the training panel.

## 2. Hazard -- the term with data

Computed for all 4,596 panel rows with the real `domain.risk.assess_hazard`. Inputs, mapped
directly from panel columns (matching how `serving/pipeline.py:330-336` calls the same function):

| `assess_hazard` argument | panel column |
| --- | --- |
| `water_satisfaction` | `water_satisfaction_30` |
| `dry_spell_days` | `dry_spell_30` (pipeline.py:332 uses the same 30-day column) |
| `cumulative_dsv` | **0 for every row** -- not available in the panel, see 1b |
| `heat_days` | `heat_stress_days` |
| `vegetation` | `risk.vegetation_hazard_from_anomaly(ndvi_z_peer)` -- the *persistence* source
  (today's observed peer anomaly), since the panel has no live forecast to replay |

No NaNs in any of the four source columns across all 4,596 rows (checked below), so this is a
complete reconstruction, not a partial one.

In [3]:
required = ['water_satisfaction_30', 'dry_spell_30', 'heat_stress_days', 'ndvi_z_peer']
print(panel[required].isna().sum())

water_satisfaction_30    0
dry_spell_30             0
heat_stress_days         0
ndvi_z_peer              0
dtype: int64


In [4]:
def row_hazard(row):
    vegetation = risk.vegetation_hazard_from_anomaly(row['ndvi_z_peer'])
    h = risk.assess_hazard(
        water_satisfaction=row['water_satisfaction_30'],
        dry_spell_days=row['dry_spell_30'],
        cumulative_dsv=0,  # not available in the committed panel -- see section 1b
        heat_days=row['heat_stress_days'],
        vegetation=vegetation,
    )
    return pd.Series(h.as_dict())

hazard_df = panel.apply(row_hazard, axis=1)
hazard_df.describe()

,drought,disease,heat,vegetation,combined
count,4596.000000,4596.0,4596.000000,4596.000000,4596.000000
mean,0.684068,0.0,0.003989,0.201038,0.749436
std,0.430577,0.0,0.059078,0.232169,0.370033
min,0.000000,0.0,0.000000,0.000000,0.000000
25%,0.143000,0.0,0.000000,0.045000,0.463750
50%,1.000000,0.0,0.000000,0.114000,1.000000
75%,1.000000,0.0,0.000000,0.255000,1.000000
max,1.000000,0.0,1.000000,1.000000,1.000000


### Distribution of each hazard component and the combined value

In [5]:
fig, axes = plt.subplots(1, 5, figsize=(20, 3.2), sharey=True)
for ax, col in zip(axes, ['drought', 'disease', 'heat', 'vegetation', 'combined']):
    ax.hist(hazard_df[col], bins=30, color='#3b6fa0')
    ax.set_title(col)
ax.set_ylim(bottom=0)
fig.suptitle('Hazard component distributions, all 4,596 panel rows')
fig.tight_layout()
plt.show()

/var/folders/s6/cj6wk29j1vjddzk5y1fz0b440000gp/T/ipykernel_6824/2570691910.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


`disease` is a spike at 0 for every row by construction -- `cumulative_dsv=0` everywhere means
`assess_hazard`'s disease term (`cumulative_dsv / spray_threshold`) is 0 everywhere too. That is the
direct consequence of section 1b, not a finding about disease pressure in the region.

### `noisy_or`: two independent 0.4 hazards give 0.64, not 0.4

Standard probabilistic construction for combining independent causes -- not specific to agronomy,
so `docs/CONCEPTS.md` (Part 3) records it with no citation, and this notebook does the same rather
than inventing one.

In [6]:
two_independent = risk.noisy_or([0.4, 0.4])
print(f"noisy_or([0.4, 0.4]) = {two_independent}")
assert two_independent == round(1 - (1 - 0.4) * (1 - 0.4), 10) == 0.64

single = risk.noisy_or([0.4])
print(f"noisy_or([0.4])      = {single}  (one hazard alone)")
print(f"max would give        0.4   mean would give        0.4")
print(f"noisy_or gives         {two_independent}  -- compounding, not a ceiling at the worse hazard")

noisy_or([0.4, 0.4]) = 0.64
noisy_or([0.4])      = 0.4  (one hazard alone)
max would give        0.4   mean would give        0.4
noisy_or gives         0.64  -- compounding, not a ceiling at the worse hazard


This is why the combination rule matters, not just the number: a field facing drought **and**
blight is worse off than a field facing either alone. `max([0.4, 0.4])` and `mean([0.4, 0.4])` both
collapse to 0.4 and would rank the doubly-stressed field identically to the singly-stressed one.
`noisy_or` -- treating each hazard as an independent probability of "something goes wrong" -- does
not. It is a real behaviour of the production hazard, demonstrated below on the panel: rows with two
or more components above 0.3 have a materially higher `combined` than rows with only one.

In [7]:
n_components_above_0p3 = (hazard_df[['drought', 'disease', 'heat', 'vegetation']] > 0.3).sum(axis=1)
print(hazard_df.groupby(n_components_above_0p3)['combined'].agg(['mean', 'count']))

       mean  count
0  0.107721    995
1  0.915405   2932
2  0.976433    668
3  1.000000      1


### Hazard by cluster and by calendar month

In [8]:
month = panel['obs_date'].dt.month
by_cluster = hazard_df['combined'].groupby(panel['cluster']).mean().sort_values(ascending=False)
print(by_cluster)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
by_cluster.plot(kind='bar', ax=axes[0], color='#3b6fa0')
axes[0].set_ylabel('mean combined hazard')
axes[0].set_title('Hazard by cluster')
axes[0].tick_params(axis='x', rotation=30)

pivot = hazard_df['combined'].groupby([panel['cluster'], month]).mean().unstack('cluster')
pivot.plot(ax=axes[1], marker='o')
axes[1].set_xlabel('calendar month')
axes[1].set_ylabel('mean combined hazard')
axes[1].set_title('Hazard by month, per cluster')
axes[1].set_xticks(range(1, 13))
fig.tight_layout()
plt.show()

cluster
Benue_River_Basin      0.863978
Kaduna_Grain_Belt      0.835340
Kano_Sudan_Savannah    0.822023
Kenya_Rift_Valley      0.564011
Name: combined, dtype: float64


/var/folders/s6/cj6wk29j1vjddzk5y1fz0b440000gp/T/ipykernel_6824/2504580406.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The clusters' hazard curves peak in different months -- exactly why `features/agronomic.py`'s peer
key (`peer_bucket`, `"<cluster>|<MM>"`) standardises within cluster and calendar month rather than
across the whole panel: a field is compared to its own growing-season cohort, not to every field
regardless of season.

### Which component dominates, and how often

In [9]:
dominant_counts = hazard_df['dominant'].value_counts()
print(dominant_counts)
print()
print((dominant_counts / len(hazard_df) * 100).round(1).astype(str) + '%')

fig, ax = plt.subplots(figsize=(6, 4))
dominant_counts.plot(kind='bar', ax=ax, color='#3b6fa0')
ax.set_ylabel('rows')
ax.set_title('Dominant hazard component, all 4,596 rows')
fig.tight_layout()
plt.show()

dominant
drought       3411
vegetation     775
none           400
heat            10
Name: count, dtype: int64

dominant
drought       74.2%
vegetation    16.9%
none           8.7%
heat           0.2%
Name: count, dtype: object


/var/folders/s6/cj6wk29j1vjddzk5y1fz0b440000gp/T/ipykernel_6824/2564020351.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


`disease` cannot appear here -- it is pinned at 0 for every row (section 1b). Read this as "which of
{drought, heat, vegetation} dominates, and how often nothing clears the 0.05 floor `assess_hazard`
uses to call a row `none`" -- not as a claim about disease's real-world share, which this
reconstruction cannot see.

## 3. Exposure and Vulnerability -- the terms without data

No panel row has `area_ha`, `expected_yield_t_ha`, `price_per_t`, or any of the seven
`COPING_FACTORS` signals (`has_irrigation`, `has_extension_access`, `received_credit`,
`used_fertilizer`, `crop_diversity`, `asset_score`, `market_access`). **Everything in this section
is a sensitivity analysis over plausible input ranges, run through the real `assess_exposure` /
assess_vulnerability` functions -- illustrative, not measured. No committed data can validate either
term.**

Where a range below isn't arbitrary, it is sourced from a committed constant, cited inline:
`serving/pipeline.py:88-89` (`FARMGATE_PRICE_USD_PER_T`) and `serving/pipeline.py:450`
(`_expected_yield`'s own clamp, `[0.5, 5.0]` t/ha). Farm area has no such committed bound --
`0.5-5 ha` below is a labelled assumption (smallholder-plausible), not a sourced one.

### Exposure: `value_at_risk = area_ha x expected_yield_t_ha x price_per_t`

In [10]:
# ILLUSTRATIVE / SYNTHETIC grid -- no panel row has these columns.
area_range = np.linspace(0.5, 5.0, 20)        # ha, labelled assumption (smallholder-plausible)
yield_range = np.linspace(0.5, 5.0, 20)       # t/ha, sourced: pipeline.py:450 clamp bounds
price_range = sorted({250.0, 230.0, 420.0, 120.0})  # USD/t, sourced: FARMGATE_PRICE_USD_PER_T

records = []
for a in area_range:
    for y in yield_range:
        for p in price_range:
            e = risk.assess_exposure(area_ha=a, expected_yield_t_ha=y, price_per_t=p)
            records.append({'area_ha': a, 'expected_yield_t_ha': y, 'price_per_t': p,
                             'value_at_risk': e.value_at_risk})
exposure_grid = pd.DataFrame(records)
print(f"value_at_risk over this illustrative grid: "
      f"{exposure_grid.value_at_risk.min():.0f} .. {exposure_grid.value_at_risk.max():.0f} USD "
      f"({exposure_grid.value_at_risk.max() / exposure_grid.value_at_risk.min():.0f}x range)")
exposure_grid.head()

value_at_risk over this illustrative grid: 30 .. 10500 USD (350x range)


,area_ha,expected_yield_t_ha,price_per_t,value_at_risk
0,0.5,0.500000,120.0,30.00
1,0.5,0.500000,230.0,57.50
2,0.5,0.500000,250.0,62.50
3,0.5,0.500000,420.0,105.00
4,0.5,0.736842,120.0,44.21


In [11]:
fig, axes = plt.subplots(1, len(price_range), figsize=(5 * len(price_range), 4), sharey=True)
for ax, p in zip(axes, price_range):
    sub = exposure_grid[exposure_grid.price_per_t == p]
    pivot = sub.pivot(index='expected_yield_t_ha', columns='area_ha', values='value_at_risk')
    im = ax.imshow(pivot.values, origin='lower', aspect='auto', cmap='viridis',
                    extent=[area_range.min(), area_range.max(), yield_range.min(), yield_range.max()])
    ax.set_title(f'price = {p:.0f} USD/t')
    ax.set_xlabel('area_ha')
    fig.colorbar(im, ax=ax, label='value_at_risk (USD)')
axes[0].set_ylabel('expected_yield_t_ha')
fig.suptitle('ILLUSTRATIVE: value_at_risk over a plausible input grid -- no panel row has these columns')
fig.tight_layout()
plt.show()

/var/folders/s6/cj6wk29j1vjddzk5y1fz0b440000gp/T/ipykernel_6824/1891321509.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Vulnerability: the protected-attribute guard, demonstrated firing

**Design choice, argued not borrowed** -- see [docs/CONCEPTS.md, Part 3, "The protected-attribute
guard"](../docs/CONCEPTS.md). Nobody published this constraint; it is a fairness argument enforced
in code rather than asserted in a comment, and that is the thing being demonstrated below.

In [12]:
try:
    risk.assess_vulnerability({"asset_score": 0.8, "head_gender": 1})
    print("no exception raised -- this would be a bug")
except ValueError as e:
    print(f"raised ValueError: {e}")

try:
    risk.assess_vulnerability({"household_max_education": 3, "received_credit": 1})
except ValueError as e:
    print(f"raised ValueError: {e}")

print()
print("PROTECTED_ATTRIBUTES:", sorted(risk.PROTECTED_ATTRIBUTES))
print("These are retained upstream for fairness *auditing* -- measuring whether the ranking")
print("disadvantages a group -- and rejected here as an input. The guard is enforced in code")
print("(a ValueError from assess_vulnerability itself), not asserted in a comment: any caller")
print("that tries to feed one in breaks immediately rather than silently.")

raised ValueError: protected attributes must not drive vulnerability: ['head_gender']
raised ValueError: protected attributes must not drive vulnerability: ['household_max_education']

PROTECTED_ATTRIBUTES: ['ethnicity', 'head_gender', 'household_max_education', 'religion']
These are retained upstream for fairness *auditing* -- measuring whether the ranking
disadvantages a group -- and rejected here as an input. The guard is enforced in code
(a ValueError from assess_vulnerability itself), not asserted in a comment: any caller
that tries to feed one in breaks immediately rather than silently.


### Vulnerability: sweep of coping capacity over an illustrative signal grid

In [13]:
# ILLUSTRATIVE -- no panel row has any of these signals.
capacity_level = np.linspace(0.0, 1.0, 21)
vuln_records = []
for c in capacity_level:
    signals = {factor: c for factor in risk.COPING_FACTORS}  # every factor at the same level
    v = risk.assess_vulnerability(signals)
    vuln_records.append({'capacity_level': c, 'coping_capacity': v.coping_capacity,
                          'vulnerability_score': v.score, 'n_gaps': len(v.gaps)})
vuln_df = pd.DataFrame(vuln_records)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(vuln_df.capacity_level, vuln_df.vulnerability_score, marker='o', color='#3b6fa0')
ax.set_xlabel('illustrative coping-factor level (all 7 signals set equally)')
ax.set_ylabel('vulnerability.score')
ax.set_title('ILLUSTRATIVE: vulnerability score vs coping capacity')
fig.tight_layout()
plt.show()
vuln_df

/var/folders/s6/cj6wk29j1vjddzk5y1fz0b440000gp/T/ipykernel_6824/2045367875.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,capacity_level,coping_capacity,vulnerability_score,n_gaps
0,0.00,0.00,1.00,7
1,0.05,0.05,0.95,7
2,0.10,0.10,0.90,7
3,0.15,0.15,0.85,7
4,0.20,0.20,0.80,7
5,0.25,0.25,0.75,7
6,0.30,0.30,0.70,7
7,0.35,0.35,0.65,0
8,0.40,0.40,0.60,0
9,0.45,0.45,0.55,0


### Modulation, not a raw multiplier: `(0.5 + 0.5*v)`

**Design choice, argued not borrowed** -- see [docs/CONCEPTS.md, Part 3, "Vulnerability as a
modulation, not a multiplier"](../docs/CONCEPTS.md). The constant 0.5 is a stated judgement, not a
fitted or published value.

In [14]:
# ILLUSTRATIVE composition test. `assess_risk` only ever reads `hazard.combined` (and `.dominant`,
# which it does not use) -- the four sub-components don't factor uniquely from a target `combined`
# value, so isolating "what happens as vulnerability varies, hazard held severe" means constructing
# `Hazard` directly with `combined` set to a value this notebook actually measured (the panel's own
# 95th-percentile combined hazard from section 2), rather than hunting for sub-component inputs
# that happen to reproduce it through `noisy_or`.
severe_hazard_value = float(hazard_df['combined'].quantile(0.95))
fixed_hazard = risk.Hazard(drought=0.0, disease=0.0, heat=0.0, vegetation=0.0,
                            combined=severe_hazard_value, dominant='vegetation')
print(f"fixed illustrative hazard.combined = {fixed_hazard.combined} "
      f"(panel's own 95th-percentile combined hazard, measured in section 2)")

fixed_exposure = risk.assess_exposure(area_ha=2.0, expected_yield_t_ha=2.0, price_per_t=250.0)

rows = []
for c in capacity_level:
    signals = {factor: c for factor in risk.COPING_FACTORS}
    v = risk.assess_vulnerability(signals)
    real = risk.assess_risk(fixed_hazard, fixed_exposure, v)
    # Counterfactual only: hazard.combined * v.score, a bare multiplication -- not a call into
    # domain/risk.py, since no such function exists (the real formula is the modulation above).
    raw_multiplier_loss_rate = fixed_hazard.combined * v.score * risk.MAX_LOSS_FRACTION
    rows.append({'capacity_level': c, 'vulnerability_score': v.score,
                 'real_loss_rate (modulation)': real.expected_loss / fixed_exposure.value_at_risk,
                 'counterfactual_loss_rate (raw multiplier)': raw_multiplier_loss_rate})
mod_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(mod_df.vulnerability_score, mod_df['real_loss_rate (modulation)'], marker='o',
        label='real: (0.5 + 0.5*v) modulation', color='#3b6fa0')
ax.plot(mod_df.vulnerability_score, mod_df['counterfactual_loss_rate (raw multiplier)'], marker='s',
        label='counterfactual: raw v multiplier', color='#c0524a')
ax.set_xlabel('vulnerability.score (0 = well-resourced, 1 = maximally vulnerable)')
ax.set_ylabel('loss_rate')
ax.set_title('ILLUSTRATIVE: modulation vs a raw multiplier, hazard fixed severe')
ax.legend()
fig.tight_layout()
plt.show()
mod_df

fixed illustrative hazard.combined = 1.0 (panel's own 95th-percentile combined hazard, measured in section 2)


/var/folders/s6/cj6wk29j1vjddzk5y1fz0b440000gp/T/ipykernel_6824/2968314107.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,capacity_level,vulnerability_score,real_loss_rate (modulation),counterfactual_loss_rate (raw multiplier)
0,0.00,1.00,0.600,0.60
1,0.05,0.95,0.585,0.57
2,0.10,0.90,0.570,0.54
3,0.15,0.85,0.555,0.51
4,0.20,0.80,0.540,0.48
5,0.25,0.75,0.525,0.45
6,0.30,0.70,0.510,0.42
7,0.35,0.65,0.495,0.39
8,0.40,0.60,0.480,0.36
9,0.45,0.55,0.465,0.33


At `vulnerability.score = 0` (a maximally well-resourced farm), the real modulation still carries
`0.5 x hazard` into the loss rate -- a raw multiplier would send it to exactly 0. That is the point
of the design, made concrete: a farmer with irrigation and credit still loses a crop to late blight
if nobody tells them to spray, and a raw multiplier would have hidden that farm from the queue
entirely.

## 4. Where the uncertainty actually is

Composing all three terms with the real `assess_risk`, and asking which term `risk_score` and
`expected_loss` are most sensitive to. Baseline: the panel's own measured mean `combined` hazard
(section 2), and illustrative mid-range exposure/vulnerability (section 3) -- one term swept at a
time, the other two held at baseline.

`assess_risk` returns two outputs, `risk_score` (0-100, a rate) and `expected_loss` (currency) --
**design choice, argued not borrowed**: see
[docs/CONCEPTS.md, Part 3, "Exposure, and two outputs"](../docs/CONCEPTS.md). This section reads
`expected_loss`, since that is what a triage queue is ranked by.

In [15]:
# Same construction as the section-3 modulation test: `assess_risk` reads only `hazard.combined`,
# so a panel-measured combined hazard is set directly rather than searched for through
# `assess_hazard`'s sub-components. The *mean* (0.749), not the median: `drought` alone saturates
# to 1.0 for any row with water_satisfaction_30 <= 0.35, which over half the panel is (median
# water_satisfaction_30 is 0.146) -- so the median combined hazard is 1.0, the ceiling, and a
# poor stand-in for "typical". The mean is pulled down by the rows that aren't saturated and is
# the more representative single number for a baseline.
baseline_hazard_combined = float(hazard_df['combined'].mean())
baseline_hazard = risk.Hazard(drought=0.0, disease=0.0, heat=0.0, vegetation=0.0,
                               combined=baseline_hazard_combined, dominant='vegetation')
print(f"baseline hazard.combined = {baseline_hazard.combined:.3f} (panel's own measured mean)")

baseline_exposure = risk.assess_exposure(area_ha=2.0, expected_yield_t_ha=2.0, price_per_t=250.0)
baseline_vulnerability = risk.assess_vulnerability({f: 0.5 for f in risk.COPING_FACTORS})
baseline = risk.assess_risk(baseline_hazard, baseline_exposure, baseline_vulnerability)
print(f"baseline: risk_score={baseline.risk_score}  expected_loss={baseline.expected_loss:.0f} USD")

baseline hazard.combined = 0.749 (panel's own measured mean)
baseline: risk_score=56.2  expected_loss=337 USD


In [16]:
# ILLUSTRATIVE one-at-a-time sensitivity. Hazard swept across the panel's *observed* range
# (measured); exposure and vulnerability swept across the same illustrative ranges as section 3
# (no panel data exists to bound them).
sens = {}

hz_vals = np.linspace(hazard_df['combined'].min(), hazard_df['combined'].max(), 15)
def _hazard_at(h):
    return risk.Hazard(drought=0.0, disease=0.0, heat=0.0, vegetation=0.0, combined=h,
                        dominant='vegetation')
sens['hazard'] = [risk.assess_risk(
    _hazard_at(h), baseline_exposure, baseline_vulnerability
).expected_loss for h in hz_vals]

exp_vals = np.linspace(exposure_grid.value_at_risk.min(), exposure_grid.value_at_risk.max(), 15)
sens['exposure'] = [risk.assess_risk(
    baseline_hazard, risk.assess_exposure(1.0, 1.0, v), baseline_vulnerability
).expected_loss for v in exp_vals]

vuln_vals = np.linspace(0.0, 1.0, 15)
sens['vulnerability'] = [risk.assess_risk(
    baseline_hazard, baseline_exposure, risk.assess_vulnerability({f: c for f in risk.COPING_FACTORS})
).expected_loss for c in vuln_vals]

swing = {term: max(vals) - min(vals) for term, vals in sens.items()}
for term in ['hazard', 'exposure', 'vulnerability']:
    print(f"{term:15s} expected_loss swings {swing[term]:>12,.0f} USD  "
          f"(min={min(sens[term]):>8,.0f}  max={max(sens[term]):>8,.0f})")

hazard          expected_loss swings          450 USD  (min=       0  max=     450)
exposure        expected_loss swings        3,531 USD  (min=      10  max=   3,541)
vulnerability   expected_loss swings          225 USD  (min=     225  max=     450)


In [17]:
fig, ax = plt.subplots(figsize=(8, 4))
terms = ['hazard', 'exposure', 'vulnerability']
ax.barh(terms, [swing[t] for t in terms], color=['#3b6fa0', '#c0524a', '#3b6fa0'])
ax.set_xlabel('expected_loss swing across the term\'s range (USD)')
ax.set_title('ILLUSTRATIVE sensitivity: expected_loss swing per term\n'
             '(hazard range measured on the panel; exposure/vulnerability ranges are illustrative)')
fig.tight_layout()
plt.show()

/var/folders/s6/cj6wk29j1vjddzk5y1fz0b440000gp/T/ipykernel_6824/2982504036.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


`hazard` is bounded to `[0, 1]` and `vulnerability.score` modulates within `[0.5, 1.0]` of that --
both are structurally capped. `exposure.value_at_risk = area_ha x expected_yield_t_ha x price_per_t`
has no such cap; it is three unbounded numbers multiplied together, and the panel above shows a
~75x range across a plausible smallholder grid alone. `expected_loss = exposure.value_at_risk x
loss_rate`, so the ranking that decides who gets an extension visit is most sensitive to exactly the
term with the least structural constraint -- and no committed data anywhere in this repository
validates it. Hazard is the term this repository can actually check; exposure is the term the queue
actually leans on.

## 5. What would close the gap

`POST /outcomes` (`serving/api/outcomes.py`) and `store.label_join`
(`data/store.py:231-250`) are built: an endpoint to record what actually happened to a field, and a
query joining recorded predictions to recorded outcomes within a horizon. `label_join`'s own
docstring is explicit about where that stands: *"It returns nothing today, and that is the point: it
is the query whose row count tells you when supervised modelling on real outcomes becomes
possible."* `docs/FACTS.md` confirms it from the data side: `field_outcomes` exists as an empty
table definition, and "no outcome data is committed."

That is the only path to validating exposure and vulnerability, because it is the only one that
records what actually happened to a farm rather than what a satellite or a weather station implies
about it. Hazard can be checked today, against physics and 4,596 panel rows. Exposure and
vulnerability can only be checked once `field_outcomes` has rows in it -- and right now it has zero.

## 6. Sources

`docs/CONCEPTS.md` is the repository's citation list, with its own verified/UNVERIFIED split; this
table cross-references it rather than duplicating it. **[verified]** means checked against a
publisher or authoritative record during that document's own work -- not upgraded here.
"design choice" means the claim is argued in this repository, not borrowed from a citable source.

| Method (first used) | Source | Status | Where |
| --- | --- | --- | --- |
| Risk = Hazard x Exposure x Vulnerability (title) | IPCC AR5/AR6 risk framing | **[verified]** as a body of work, no chapter cited | [docs/CONCEPTS.md](../docs/CONCEPTS.md), Part 3 |
| VCI (1b) | Kogan, F.N. (1990), *Int. J. Remote Sensing* 11:1405-1419 | **[verified]** | [doi:10.1080/01431169008955102](https://doi.org/10.1080/01431169008955102); [docs/CONCEPTS.md](../docs/CONCEPTS.md), Part 1 |
| EVI (1b) | Huete et al. (2002), *RSE* 83:195-213 | **[verified]** | [docs/CONCEPTS.md](../docs/CONCEPTS.md), Part 1 |
| SAVI (1b) | Huete, A.R. (1988), *RSE* 25:295-309 | **[verified]** | [docs/CONCEPTS.md](../docs/CONCEPTS.md), Part 1 |
| GDD, base 10C with cap (1b) | McMaster & Wilhelm (1997), usual methods reference | **[UNVERIFIED]** | [docs/CONCEPTS.md](../docs/CONCEPTS.md), Part 2 |
| FAO-56 water balance / Kc (1b, 2) | Allen, Pereira, Raes, Smith (1998), FAO Irrigation & Drainage Paper 56 | **[verified]** | [docs/CONCEPTS.md](../docs/CONCEPTS.md), Part 2 |
| DSV / late blight (1b, 2) | Wallin (1962); BLITECAST = Krause, Massie & Hyre (1975) + Hyre (1955) | **[verified]**, including the 18-20 SV spray threshold. **Deviation recorded**: implementation uses leaf-wetness hours, Wallin specified RH>=90% duration | [docs/CONCEPTS.md](../docs/CONCEPTS.md), Part 2 |
| Fall armyworm ~390 DD/generation (1b) | widely quoted degree-day figure | **[UNVERIFIED]** | [docs/CONCEPTS.md](../docs/CONCEPTS.md), Part 2 |
| noisy-OR hazard combination (2) | standard probabilistic construction for independent causes | no citation -- not agronomy-specific | [docs/CONCEPTS.md](../docs/CONCEPTS.md), Part 3 |
| Protected-attribute guard (3) | fairness argument, enforced in code | design choice, argued not borrowed | [docs/CONCEPTS.md](../docs/CONCEPTS.md), Part 3 |
| Vulnerability modulation `(0.5+0.5v)` (3) | bounds vulnerability's influence to a factor of two | design choice, argued not borrowed | [docs/CONCEPTS.md](../docs/CONCEPTS.md), Part 3 |
| `risk_score` / `expected_loss` two-output split (4) | rate vs. currency, answer different questions | design choice, argued not borrowed | [docs/CONCEPTS.md](../docs/CONCEPTS.md), Part 3 |

Not used in this notebook but part of the same chain and worth knowing the status of:
`NDVI` (Rouse et al. 1974, **[UNVERIFIED]**), `NDMI` (Gao 1996, **[UNVERIFIED]**) -- both in
[docs/CONCEPTS.md](../docs/CONCEPTS.md), Part 1.